## **Install the Required Libraries**

In [1]:
!pip install -U 'langchain>=0.2.0,<0.3.0' langchain-google-genai langchain-community pypdf faiss-cpu langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usu

## **Import Dependencies & Setup API Key**

In [ ]:
import os
import json
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [ ]:
# Set your free Gemini API key here
os.environ["GOOGLE_API_KEY"] = "google_api"

In [ ]:
pdf_path = "/content/Doc 1.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages from the PDF.")

Loaded 1 pages from the PDF.


## **Split the Document**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)
docs = text_splitter.split_documents(documents)

print(f"Split the document into {len(docs)} text chunks.")

Split the document into 4 text chunks.


## **Generate Embeddings & Vector Store**


In [ ]:
# Initialize Google's Embedding Model
try:
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
except Exception as e:
    print(f"Error initializing embedding model: {e}")
    # Fallback or exit if necessary
    raise

# Create the FAISS Vector Store
vectorstore = FAISS.from_documents(docs, embeddings)

print("Local FAISS vector store created successfully!")

Local FAISS vector store created successfully!


In [ ]:
import os
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Listing available generative models:")
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(f"  {m.name} (Embeddings)")
    elif "generateContent" in m.supported_generation_methods:
        print(f"  {m.name} (Generative)")

Listing available generative models:
  models/gemini-2.5-flash (Generative)
  models/gemini-2.5-pro (Generative)
  models/gemini-2.0-flash (Generative)
  models/gemini-2.0-flash-001 (Generative)
  models/gemini-2.0-flash-lite-001 (Generative)
  models/gemini-2.0-flash-lite (Generative)
  models/gemini-2.5-flash-preview-tts (Generative)
  models/gemini-2.5-pro-preview-tts (Generative)
  models/gemma-4-26b-a4b-it (Generative)
  models/gemma-4-31b-it (Generative)
  models/gemini-flash-latest (Generative)
  models/gemini-flash-lite-latest (Generative)
  models/gemini-pro-latest (Generative)
  models/gemini-2.5-flash-lite (Generative)
  models/gemini-2.5-flash-image (Generative)
  models/gemini-3-pro-preview (Generative)
  models/gemini-3-flash-preview (Generative)
  models/gemini-3.1-pro-preview (Generative)
  models/gemini-3.1-pro-preview-customtools (Generative)
  models/gemini-3.1-flash-lite-preview (Generative)
  models/gemini-3.1-flash-lite (Generative)
  models/gemini-3-pro-image-pre

## **Set up the Gemini LLM**

In [ ]:
# Initialize the Gemini Model
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0.3 # Keeps the model factual and precise
)

## **Create the Prompt and Generation Logic**

In [ ]:
prompt_template = """
You are an expert quiz creator. Use the following pieces of context to generate exactly 3 Multiple Choice Questions (MCQs) about the topic: {topic}.

You MUST format your response as a valid JSON array of objects. Do not add any text before or after the JSON.

Example format:
[
    {{
        "question": "What is the capital of France?",
        "choices": ["A. London", "B. Paris", "C. Berlin", "D. Rome"],
        "answer": "B. Paris"
    }}
]

Context:
{context}

Generate the JSON array now:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "topic"]
)

def generate_mcqs(topic_query):
    # 1. Retrieve the most relevant chunks from FAISS
    relevant_docs = vectorstore.similarity_search(topic_query, k=3)

    # 2. Combine the text from the retrieved chunks
    context_text = "\n".join([doc.page_content for doc in relevant_docs])

    # 3. Create the LangChain pipeline
    chain = prompt | llm

    # 4. Invoke the model
    response = chain.invoke({"context": context_text, "topic": topic_query})

    # Gemini returns the text inside an AIMessage object
    return response.content

# Test it out!
topic = "India's economy and history"
print(f"Generating questions about: {topic}...\n")

raw_response = generate_mcqs(topic)
print(raw_response)

Generating questions about: India's economy and history...

[
    {
        "question": "Around which year did the Indus Valley Civilization flourish in the northwestern part of the Indian subcontinent?",
        "choices": [
            "A. 1500 BCE",
            "B. 2500 BCE",
            "C. 3500 BCE",
            "D. 500 BCE"
        ],
        "answer": "B. 2500 BCE"
    },
    {
        "question": "On what date did India gain its independence from British colonial rule?",
        "choices": [
            "A. August 15, 1947",
            "B. August 15, 1950",
            "C. January 26, 1947",
            "D. October 2, 1947"
        ],
        "answer": "A. August 15, 1947"
    },
    {
        "question": "How has India's economy transitioned over time according to the text?",
        "choices": [
            "A. From a service-oriented to a purely agrarian economy",
            "B. From an agrarian to a service-oriented and industrialized economy",
            "C. From a manu

## **Display the Final Output**

In [ ]:
def extract_json(raw_text):
    try:
        # Strip markdown formatting if the LLM includes it
        match = re.search(r'\[\s*\{.*?\}\s*\]', raw_text, re.DOTALL)
        if match:
            clean_json = match.group(0)
            return json.loads(clean_json)
        else:
            return json.loads(raw_text)
    except Exception as e:
        print("Failed to parse JSON. Raw output was:")
        print(raw_text)
        return None

parsed_mcqs = extract_json(raw_response)

if parsed_mcqs:
    print(" YOUR GENERATED MCQs \n")
    for i, q in enumerate(parsed_mcqs, 1):
        print(f"Q{i}: {q['question']}")
        for choice in q['choices']:
            print(f"   {choice}")
        print(f" Correct Answer: {q['answer']}\n")

 YOUR GENERATED MCQs 

Q1: Around which year did the Indus Valley Civilization flourish in the northwestern part of the Indian subcontinent?
   A. 1500 BCE
   B. 2500 BCE
   C. 3500 BCE
   D. 500 BCE
 Correct Answer: B. 2500 BCE

Q2: On what date did India gain its independence from British colonial rule?
   A. August 15, 1947
   B. August 15, 1950
   C. January 26, 1947
   D. October 2, 1947
 Correct Answer: A. August 15, 1947

Q3: How has India's economy transitioned over time according to the text?
   A. From a service-oriented to a purely agrarian economy
   B. From an agrarian to a service-oriented and industrialized economy
   C. From a manufacturing to a strictly agricultural economy
   D. From an industrialized to a traditional barter economy
 Correct Answer: B. From an agrarian to a service-oriented and industrialized economy

